CREATING A DATASET

In [5]:
!pip install snowflake-connector-python python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached asn1crypto-1.5.1-py2.py3-none-any.whl.metadata (13 kB)
  Using cached pyjwt-2.12.1-py3-none-any.whl.metadata (4.1 kB)
  Using cached charset_normalizer-3.4.7-cp311-cp311-macosx_10_9_universal2.whl.metadata (40 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 12.1 MB/s  0:00:01m0:00:010:01
Using cached asn1crypto-1.5.1-py2.py3-none-any.whl (105 kB)
Using cached charset_normalizer-3.4.7-cp311-cp311-macosx_10_9_universal2.whl (309 kB)
Using cached filelock-3.29.0-py3-none-any.whl (39 kB)
Using cached pyjwt-2.12.1-py3-none-any.whl (29 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
print('Hello')

Hello


In [15]:
import snowflake.connector
from dotenv import load_dotenv
import os
import json
from pathlib import Path

def get_snowflake_connection():
    """Connects to Snowflake with MFA and sets the database/schema context."""
    load_dotenv()
    totp_code = input("Enter your TOTP code (6 digits): ")
        
    conn = snowflake.connector.connect(
        user=os.getenv('SNOWFLAKE_USER'),
        password=os.getenv('SNOWFLAKE_PASSWORD'),
        account=os.getenv('SNOWFLAKE_ACCOUNT'),
        passcode=totp_code
    )
    
    cursor = conn.cursor()
    print("Connected and contexts set to DEV.AI_QUOTE_INTAKE")
    return conn, cursor

conn, cursor = get_snowflake_connection()
cursor.execute("USE WAREHOUSE CHIPMUNK")

Connected and contexts set to DEV.AI_QUOTE_INTAKE


In [17]:
import pandas as pd
import json

cursor.execute("USE WAREHOUSE CHIPMUNK")


CORTEX_PROMPT = """Generate {n} pairs of (natural language question, ADQL query) for the Gaia DR3 database.
Table: gaiadr3.gaia_source
Complexity: {complexity}

Rules:
- Simple:  1 table, basic WHERE filters
- Medium:  2+ conditions, aggregations, or ORDER BY
- Complex: JOINs, subqueries, or window functions

Return ONLY a JSON array, no explanation:
[
  {{"question": "...", "sql": "...", "complexity": "{complexity}"}},
  ...
]"""

def generate_pairs(cursor, complexity, n=10, model='mistral-large'):
    prompt = CORTEX_PROMPT.format(n=n, complexity=complexity)
    cursor.execute(
        "SELECT SNOWFLAKE.CORTEX.COMPLETE(%s, %s)",
        (model, prompt)
    )
    raw = cursor.fetchone()[0]
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    return json.loads(match.group()) if match else []

# ── generate across all complexity levels ─────────────────────────────────────
import re

rows = []
for complexity in ['Simple', 'Medium', 'Complex']:
    print(f"Generating {complexity}...")
    pairs = generate_pairs(cursor, complexity, n=10)
    rows.extend(pairs)
    print(f"  → {len(pairs)} pairs")

df_generated = pd.DataFrame(rows)
df_generated.to_csv('queries_cortex.csv', index=False)
print(f"\nTotal: {len(df_generated)} pairs saved to queries_cortex.csv")
print(df_generated['complexity'].value_counts())

Generating Simple...


KeyboardInterrupt: 

In [ ]:
# ── only the intents that map to truly simple queries ────────────────────────
SIMPLE_INTENTS = ['cone_search', 'stellar_population', 'nearest_neighbours', 'hr_diagram']


def build_prompt(intent: str, n: int) -> str:
    spec = INTENT_SPECS[intent]
    return f"""You are generating a SIMPLE evaluation dataset for a text-to-ADQL pipeline that targets the Gaia DR3 archive.

Generate {n} diverse (question, ADQL) pairs for the intent: {intent}
Description: {spec['note']}

You MUST follow these rules strictly:

1. Use ONLY these column names: {', '.join(VALID_COLUMNS)}
2. The table is always: gaiadr3.gaia_source
3. Every ADQL query MUST have:
   - SELECT TOP <N> ... (N between 100 and 1000)
   - A WHERE clause (never omit it)
   - ORDER BY random_index (except nearest_neighbours which uses ORDER BY parallax DESC)
4. For cone-required intents, include: 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', <ra>, <dec>, <radius>))
   - Use real coordinates of well-known objects (Pleiades 56.75/24.12, Orion 83.8/-5.4,
     Andromeda 10.68/41.27, LMC 80.9/-69.8, Galactic Centre 266.4/-29.0,
     Omega Centauri 201.7/-47.5, Barnard's Star 269.45/4.69)
   - Radius between 0.5 and 2.0 degrees
5. KEEP IT SIMPLE — exactly ONE filter beyond the spatial cone (or one filter total for sky-wide).
   No JOINs. No BETWEEN. No multiple range filters. No complex WHERE composition.
6. Suggested simple filters:
   - parallax > 5
   - phot_g_mean_mag < 12
   - teff_gspphot > 5000
   - bp_rp > 1.5
7. Questions should be short and natural — one clear ask, no compound requests.
   Never reveal the intent name or column names in the question.

Return ONLY a JSON array, no markdown fences, no explanation:
[
  {{"question": "...", "sql": "SELECT TOP ...", "intent": "{intent}", "complexity": "Simple"}},
  ...
]
"""


# ── run ───────────────────────────────────────────────────────────────────────

MODEL      = 'mistral-large'
PER_INTENT = 3

all_rows = []
for intent in SIMPLE_INTENTS:
    print(f"Generating {PER_INTENT} pairs for '{intent}' …")
    pairs = generate_for_intent(cursor, intent, PER_INTENT, MODEL)
    print(f"  → got {len(pairs)} pairs")
    all_rows.extend(pairs)

df = pd.DataFrame(all_rows)
df.to_csv('gaia_eval_dataset.csv', index=False)

print(f"\nTotal: {len(df)} pairs")
print(df['intent'].value_counts())

Generating 5 pairs for 'cone_search' …


NameError: name 'generate_for_intent' is not defined